# C2-linear-models — Practice p23

**Type:** constrained coding · **Difficulty:** core · **Concepts:** ols-rank-identifiability-and-pseudoinverse

Implement \`ols_pinv(X, y)\` for ordinary least squares at any rank.
Accept finite numeric \`X (n, p)\` and \`y (n,)\` with matching,
nonempty dimensions; raise \`ValueError\` before computing for malformed
or non-finite input. Do not mutate inputs.

Return float \`beta (p,)\` equal to $\beta^+=X^+y$, the
minimum-Euclidean-norm least-squares coefficient. Make exactly one
\`np.linalg.pinv(X)\` call and multiply its returned matrix by $y$.
Rejected input makes zero pinv calls.

RTOL is used only for coefficient, projection (including its residual),
and nullspace/minimum-norm comparisons with \`ATOL = 1e-10\`,
\`RTOL = 1e-10\`; it does not
apply to the orthogonality-to-zero check. That check uses exactly
\`ATOL + BACKWARD_SAFETY * eps * max(1, effective_condition) * max(1, problem_scale)\`,
where \`eps = np.finfo(float).eps\`, \`BACKWARD_SAFETY = 64.0\`, and
\`problem_scale = ||X.T||_inf ||X beta - y||_inf\`. The effective
condition number is the largest singular value divided by the smallest
singular value strictly above the numerical-rank cutoff
\`eps * max(X.shape) * largest_singular_value\`, capped at \`1 / eps\`.
When numerical rank is zero, \`effective_condition = 1.0\`.

The function and its helpers have a constrained grading namespace.
They may load the exact \`np\` global, the provided \`ATOL\`, \`RTOL\`,
\`FLOAT_EPS\`, and \`BACKWARD_SAFETY\` constants, main-module helper
functions, and only these builtins: \`ValueError\`, \`float\`, \`int\`,
\`bool\`, \`len\`, \`all\`, and \`any\`. Permitted NumPy/data
attribute names are \`array\`, \`asarray\`, \`isfinite\`,
\`issubdtype\`, \`number\`, \`floating\`, \`integer\`, \`linalg\`,
\`pinv\`, \`ndim\`, \`shape\`, \`size\`, \`dtype\`, \`astype\`,
\`all\`, \`copy\`, and \`T\`. Helpers are recursively audited
under the same allowlist. Custom classes, callable objects, partials,
bound methods, extra modules, unknown attributes, imports, and runtime
namespace/reflection routes such as \`globals\`, \`locals\`, \`vars\`,
\`eval\`, \`exec\`, \`compile\`, \`__import__\`, \`getattr\`,
\`setattr\`, and \`delattr\` are not permitted. This is a grading
bytecode namespace contract, not a security sandbox.

**Banned inside the function and any helper it calls (zero points):**
\`np.linalg.inv\`, \`np.linalg.solve\`, \`np.linalg.lstsq\`,
\`getattr\`, \`__dict__\`, any spelling of \`sklearn\` or
\`statsmodels\`, loops, comprehensions, recursion, or saved aliases to
forbidden routines.

The immutable checker patches forbidden APIs, audits nested code and
referenced globals/defaults/closures, uses full-rank, deficient, and
scaled fixtures, and verifies that output propagates from the returned
pseudoinverse matrix rather than a dummy call.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 1e-10
FLOAT_EPS = np.finfo(float).eps
BACKWARD_SAFETY = 64.0


def ols_pinv(X, y):
    X_array = np.asarray(X)
    y_array = np.asarray(y)
    X_numeric = (np.issubdtype(X_array.dtype, np.floating) or
                 np.issubdtype(X_array.dtype, np.integer))
    y_numeric = (np.issubdtype(y_array.dtype, np.floating) or
                 np.issubdtype(y_array.dtype, np.integer))
    if (X_array.ndim != 2 or y_array.ndim != 1 or
            X_array.size == 0 or y_array.size == 0 or
            X_array.shape[0] != y_array.shape[0] or
            not X_numeric or not y_numeric or
            not np.isfinite(X_array).all() or
            not np.isfinite(y_array).all()):
        raise ValueError("invalid OLS input")
    X_float = X_array.astype(float, copy=True)
    y_float = y_array.astype(float, copy=True)
    pseudoinverse = np.linalg.pinv(X_float)
    beta = pseudoinverse @ y_float
    beta_float = np.asarray(beta).astype(float, copy=True)
    if (beta_float.ndim != 1 or
            beta_float.shape[0] != X_float.shape[1] or
            not np.isfinite(beta_float).all()):
        raise ValueError("pseudoinverse returned an invalid coefficient vector")
    return beta_float

## Immutable contract check — do not edit

The checker enforces exact pinv input, one-call data flow, forbidden-route
isolation, recursive alias inspection, scale-aware semantics, and
minimum norm on independent full- and deficient-rank fixtures.

In [ ]:
import dis
import builtins
import inspect
import types

_ORIGINAL_PINV_P23 = np.linalg.pinv
_ORIGINAL_INV_P23 = np.linalg.inv
_ORIGINAL_SOLVE_P23 = np.linalg.solve
_ORIGINAL_LSTSQ_P23 = np.linalg.lstsq
_FORBIDDEN_FUNCS_P23 = (
    _ORIGINAL_INV_P23, _ORIGINAL_SOLVE_P23, _ORIGINAL_LSTSQ_P23,
)
assert FLOAT_EPS == np.finfo(float).eps
assert BACKWARD_SAFETY == 64.0
_APPROVED_HELPER_MODULE_P23 = "__main__"
_SAFE_CONSTANT_TYPES_P23 = (
    type(None), bool, int, float, complex, str, bytes,
)
_ALLOWED_BUILTIN_NAMES_P23 = {
    "ValueError", "float", "int", "bool", "len", "all", "any",
}
_ALLOWED_GLOBAL_CONSTANTS_P23 = {
    "ATOL", "RTOL", "FLOAT_EPS", "BACKWARD_SAFETY",
}
_ALLOWED_ATTR_NAMES_P23 = {
    "array", "asarray", "isfinite", "issubdtype", "number",
    "floating", "integer", "linalg", "pinv", "ndim", "shape",
    "size", "dtype", "astype", "all", "copy", "T",
}
_BANNED_NAMESPACE_NAMES_P23 = {
    "globals", "locals", "vars", "eval", "exec", "compile",
    "__import__", "getattr", "setattr", "delattr", "dir",
    "hasattr",
}
_SAFE_BUILTINS_P23 = tuple(
    vars(builtins)[name] for name in _ALLOWED_BUILTIN_NAMES_P23
)


def _audit_value_p23(value, seen, pending_functions):
    marker = id(value)
    if marker in seen:
        return
    seen.add(marker)
    assert all(value is not item for item in _FORBIDDEN_FUNCS_P23)

    if value is np:
        return
    if isinstance(value, types.FunctionType):
        assert value.__module__ == _APPROVED_HELPER_MODULE_P23
        pending_functions.append(value)
    elif isinstance(value, dict):
        for key, item in value.items():
            _audit_value_p23(key, seen, pending_functions)
            _audit_value_p23(item, seen, pending_functions)
    elif isinstance(value, (tuple, list, set, frozenset)):
        for item in value:
            _audit_value_p23(item, seen, pending_functions)
    elif type(value) in _SAFE_CONSTANT_TYPES_P23:
        return
    else:
        assert any(value is item for item in _SAFE_BUILTINS_P23)


_pending_functions_p23 = [ols_pinv]
_seen_functions_p23 = set()
_codes_p23 = []
while _pending_functions_p23:
    _function_p23 = _pending_functions_p23.pop()
    if id(_function_p23) in _seen_functions_p23:
        continue
    assert _function_p23.__module__ == _APPROVED_HELPER_MODULE_P23
    _seen_functions_p23.add(id(_function_p23))
    _defaults_p23 = (
        tuple(_function_p23.__defaults__ or ())
        + tuple((_function_p23.__kwdefaults__ or {}).values())
    )
    for _value_p23 in _defaults_p23:
        _audit_value_p23(_value_p23, set(), _pending_functions_p23)
    for _cell_p23 in (_function_p23.__closure__ or ()):
        _audit_value_p23(_cell_p23.cell_contents, set(), _pending_functions_p23)

    _code_pending_p23 = [_function_p23.__code__]
    while _code_pending_p23:
        _code_p23 = _code_pending_p23.pop()
        _codes_p23.append(_code_p23)
        _code_pending_p23.extend(
            item for item in _code_p23.co_consts
            if isinstance(item, types.CodeType)
        )
        _names_p23 = {name.lower() for name in _code_p23.co_names}
        assert not (_names_p23 & {
            "inv", "solve", "lstsq", "getattr", "__dict__",
            "sklearn", "statsmodels",
        })
        assert _function_p23.__name__ not in _code_p23.co_names
        _ops_p23 = {item.opname for item in dis.get_instructions(_code_p23)}
        assert not (_ops_p23 & {
            "FOR_ITER", "IMPORT_NAME", "LOAD_BUILD_CLASS",
            "STORE_ATTR", "DELETE_ATTR",
        })
        assert not any(name.startswith("JUMP_BACKWARD") for name in _ops_p23)
        for _instruction_p23 in dis.get_instructions(_code_p23):
            _name_p23 = _instruction_p23.argval
            if _instruction_p23.opname in {"LOAD_ATTR", "LOAD_METHOD"}:
                assert _name_p23 in _ALLOWED_ATTR_NAMES_P23
            elif _instruction_p23.opname in {"LOAD_GLOBAL", "LOAD_NAME"}:
                assert _name_p23 not in _BANNED_NAMESPACE_NAMES_P23
                assert not (
                    _name_p23.startswith("__")
                    or _name_p23.endswith("__")
                )
                if _name_p23 == "np":
                    assert _function_p23.__globals__.get(_name_p23) is np
                elif _name_p23 in _ALLOWED_GLOBAL_CONSTANTS_P23:
                    _constant_p23 = _function_p23.__globals__[_name_p23]
                    assert type(_constant_p23) in _SAFE_CONSTANT_TYPES_P23
                elif _name_p23 in _ALLOWED_BUILTIN_NAMES_P23:
                    if _name_p23 in _function_p23.__globals__:
                        assert (
                            _function_p23.__globals__[_name_p23]
                            is vars(builtins)[_name_p23]
                        )
                else:
                    _helper_p23 = _function_p23.__globals__[_name_p23]
                    assert isinstance(_helper_p23, types.FunctionType)
                    _audit_value_p23(
                        _helper_p23, set(), _pending_functions_p23,
                    )

try:
    _source_p23 = inspect.getsource(ols_pinv).lower()
except (OSError, TypeError):
    _source_p23 = ""
assert all(token not in _source_p23 for token in (
    "np.linalg.inv(", "np.linalg.solve(", "np.linalg.lstsq(",
    "getattr(", "__dict__", "sklearn", "statsmodels",
))

_X_full_p23 = np.array([
    [1.0, -3.0],
    [1.0, -1.0],
    [1.0, 2.0],
    [1.0, 5.0],
    [1.0, 8.0],
])
_y_full_p23 = np.array([-2.0, 0.4, 3.7, 8.2, 11.5])
_X_def_p23 = np.array([
    [1.0, -2.0, -4.0],
    [1.0, -1.0, -2.0],
    [1.0, 1.0, 2.0],
    [1.0, 3.0, 6.0],
    [1.0, 6.0, 12.0],
])
_y_def_p23 = np.array([-1.0, 0.2, 2.5, 5.3, 9.1])
_z_def_p23 = np.array([0.0, -2.0, 1.0])
_X_ill_p23 = np.array([
    [1.0, 1.0],
    [1.0, 1.0 + 1e-9],
    [1.0, 1.0 - 1e-9],
    [1.0, 1.0 + 2e-9],
])
_y_ill_p23 = np.arange(4.0)
_X_zero_p23 = np.zeros((4, 3))
_y_zero_p23 = np.array([2.0, -1.0, 4.0, 3.0])
_z_zero_p23 = np.array([1.0, -2.0, 3.0])
_fixtures_p23 = (
    (_X_full_p23, _y_full_p23, None),
    (_X_def_p23, _y_def_p23, _z_def_p23),
    (_X_def_p23 * 1e8, _y_def_p23 * 1e8, _z_def_p23),
    (_X_ill_p23, _y_ill_p23, None),
    (_X_zero_p23, _y_zero_p23, _z_zero_p23),
)


def _invoke_p23(X, y, pinv_replacement):
    forbidden_calls = []

    def forbid(name):
        def blocked(*args, **kwargs):
            forbidden_calls.append(name)
            raise AssertionError(f"forbidden route called: {name}")
        return blocked

    np.linalg.pinv = pinv_replacement
    np.linalg.inv = forbid("inv")
    np.linalg.solve = forbid("solve")
    np.linalg.lstsq = forbid("lstsq")
    try:
        result = ols_pinv(X, y)
    finally:
        np.linalg.pinv = _ORIGINAL_PINV_P23
        np.linalg.inv = _ORIGINAL_INV_P23
        np.linalg.solve = _ORIGINAL_SOLVE_P23
        np.linalg.lstsq = _ORIGINAL_LSTSQ_P23
    assert forbidden_calls == []
    return result


for _X_p23, _y_p23, _z_p23 in _fixtures_p23:
    _expected_p23 = _ORIGINAL_PINV_P23(_X_p23) @ _y_p23
    _pinv_calls_p23 = []

    def counted_pinv(a, *args, **kwargs):
        _pinv_calls_p23.append(np.array(a, copy=True))
        return _ORIGINAL_PINV_P23(a, *args, **kwargs)

    _X_before_p23 = _X_p23.copy()
    _y_before_p23 = _y_p23.copy()
    _beta_p23 = _invoke_p23(_X_p23, _y_p23, counted_pinv)
    assert len(_pinv_calls_p23) == 1
    assert np.array_equal(_pinv_calls_p23[0], _X_p23)
    assert isinstance(_beta_p23, np.ndarray)
    assert _beta_p23.shape == (_X_p23.shape[1],)
    assert np.issubdtype(_beta_p23.dtype, np.floating)
    assert np.isfinite(_beta_p23).all()
    assert np.array_equal(_X_p23, _X_before_p23)
    assert np.array_equal(_y_p23, _y_before_p23)
    assert np.allclose(
        _beta_p23, _expected_p23, atol=ATOL, rtol=RTOL,
    )
    _pred_p23 = _X_p23 @ _beta_p23
    _expected_pred_p23 = _X_p23 @ _expected_p23
    _resid_p23 = _pred_p23 - _y_p23
    _expected_resid_p23 = _expected_pred_p23 - _y_p23
    assert np.allclose(
        _pred_p23, _expected_pred_p23, atol=ATOL, rtol=RTOL,
    )
    assert np.allclose(
        _resid_p23, _expected_resid_p23, atol=ATOL, rtol=RTOL,
    )
    _orth_gap_p23 = np.linalg.norm(
        _X_p23.T @ _resid_p23, ord=np.inf,
    )
    _orth_scale_p23 = (
        np.linalg.norm(_X_p23.T, ord=np.inf)
        * np.linalg.norm(_resid_p23, ord=np.inf)
    )
    _singular_p23 = np.linalg.svd(_X_p23, compute_uv=False)
    _rank_cutoff_p23 = (
        FLOAT_EPS
        * max(_X_p23.shape)
        * _singular_p23[0]
    )
    _nonzero_p23 = _singular_p23[_singular_p23 > _rank_cutoff_p23]
    if _nonzero_p23.size == 0:
        _effective_cond_p23 = 1.0
        assert np.array_equal(
            _beta_p23, np.zeros(_X_p23.shape[1]),
        )
        assert np.array_equal(
            _pred_p23, np.zeros(_X_p23.shape[0]),
        )
        assert np.array_equal(_resid_p23, -_y_p23)
        assert _orth_gap_p23 == 0.0
    else:
        _effective_cond_p23 = min(
            _singular_p23[0] / _nonzero_p23[-1],
            1 / FLOAT_EPS,
        )
    _orth_bound_p23 = (
        ATOL
        + BACKWARD_SAFETY
        * FLOAT_EPS
        * max(1.0, _effective_cond_p23)
        * max(1.0, _orth_scale_p23)
    )
    assert np.isfinite(_orth_bound_p23)
    assert _orth_gap_p23 <= _orth_bound_p23
    if _z_p23 is not None:
        _null_gap_p23 = np.linalg.norm(_X_p23 @ _z_p23, ord=np.inf)
        _null_scale_p23 = (
            np.linalg.norm(_X_p23, ord=np.inf)
            * np.linalg.norm(_z_p23, ord=np.inf)
        )
        assert _null_gap_p23 <= ATOL + RTOL * _null_scale_p23
        for _scale_p23 in (-3.0, -0.5, 0.75, 4.0):
            _shifted_p23 = _beta_p23 + _scale_p23 * _z_p23
            assert np.allclose(
                _X_p23 @ _shifted_p23,
                _pred_p23,
                atol=ATOL, rtol=RTOL,
            )
            _norm_bound_p23 = (
                ATOL + RTOL * max(
                    np.linalg.norm(_beta_p23),
                    np.linalg.norm(_shifted_p23),
                )
            )
            assert np.linalg.norm(_beta_p23) <= (
                np.linalg.norm(_shifted_p23) + _norm_bound_p23
            )

_flow_matrix_p23 = np.arange(
    _X_full_p23.shape[1] * _X_full_p23.shape[0], dtype=float,
).reshape(_X_full_p23.shape[1], _X_full_p23.shape[0])
_flow_calls_p23 = []


def flow_pinv_p23(a, *args, **kwargs):
    _flow_calls_p23.append(np.array(a, copy=True))
    return _flow_matrix_p23.copy()


_flow_beta_p23 = _invoke_p23(_X_full_p23, _y_full_p23, flow_pinv_p23)
assert len(_flow_calls_p23) == 1
assert np.array_equal(_flow_calls_p23[0], _X_full_p23)
assert np.array_equal(_flow_beta_p23, _flow_matrix_p23 @ _y_full_p23)

_rejected_p23 = (
    (np.ones(3), np.ones(3)),
    (np.ones((3, 2)), np.ones((3, 1))),
    (np.ones((3, 2)), np.ones(2)),
    (np.empty((0, 2)), np.empty(0)),
    (np.array([[1.0, np.inf], [1.0, 2.0]]), np.ones(2)),
)
for _X_bad_p23, _y_bad_p23 in _rejected_p23:
    _pinv_calls_p23 = []

    def unexpected_pinv_p23(a, *args, **kwargs):
        _pinv_calls_p23.append(a)
        return _ORIGINAL_PINV_P23(a, *args, **kwargs)

    try:
        _invoke_p23(_X_bad_p23, _y_bad_p23, unexpected_pinv_p23)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input must raise ValueError")
    assert _pinv_calls_p23 == []

### Answer check

The immutable checker audits the direct helper-free namespace, rejects malformed inputs before any pseudoinverse call, instruments exactly one `np.linalg.pinv(X)` call and return-matrix data flow, and blocks inverse, solve, and least-squares alternatives. It verifies full-rank, deficient-rank, scaled, ill-conditioned, and rank-zero fixtures, including projection, residual orthogonality with the prescribed backward bound, nullspace invariance, and minimum Euclidean norm.